# Phát Hiện Ngã (Fall Detection) Dựa Trên Gia Tốc Acc

Chỉ sử dụng 3 đặc trưng: `AccX`, `AccY`, `AccZ`.

Ngưỡng phân loại cố định là `DECISION_THRESHOLD = 0.5`: xác suất **> 0.5**
được dự đoán là Ngã, còn lại là Bình thường (cùng quy tắc với metric Keras).
Validation dùng để chọn checkpoint và điều chỉnh learning rate, không tìm ngưỡng phân loại.

Quy trình: đọc dữ liệu → chia train/validation theo subject → tạo window và chuẩn hóa
→ huấn luyện → fine-tuning → đánh giá model fine-tuned trên Sample_Test
→ xuất TFLite, scaler, metadata và C/C++ với cùng ngưỡng 0.5.

`FALL_RATIO_THRESHOLD = 0.20` là tỷ lệ frame Ngã để **gán nhãn window**,
khác với ngưỡng phân loại xác suất của model và được giữ nguyên.
Chạy các cell từ trên xuống để tạo lại kết quả và các file model.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

import tensorflow as tf
from tensorflow.keras import Model, Input, layers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from IPython.display import display

np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow Version:", tf.__version__)


In [ ]:
from pathlib import Path


# Tỷ lệ frame Ngã tối thiểu để gán nhãn window; không phải ngưỡng dự đoán.
FALL_RATIO_THRESHOLD = 0.20
# Cố định cho metric Keras, đánh giá và export; không chọn từ dữ liệu.
DECISION_THRESHOLD = 0.5


def resolve_dataset_path(*relative_candidates):
    """Resolve dataset paths whether the notebook runs from workspace or project root."""
    candidates = [Path.cwd() / candidate for candidate in relative_candidates]
    candidates.extend(Path.cwd().parent / candidate for candidate in relative_candidates)
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched = ", ".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Không tìm thấy dataset. Đã tìm: {searched}")


def load_imu_dataset(data_path):
    """Read acceleration data from every CSV and add leakage-safe identifiers."""
    data_path = Path(data_path)
    csv_files = sorted(data_path.rglob("*.csv"))
    print(f"Đang đọc {len(csv_files)} file CSV từ: {data_path}")

    required_cols = {"FrameCounter", "AccX", "AccY", "AccZ", "FallCheck"}
    dfs = []
    for file_path in csv_files:
        df = pd.read_csv(file_path, usecols=lambda col: col in required_cols)
        missing_cols = required_cols.difference(df.columns)
        if missing_cols:
            raise ValueError(f"{file_path} thiếu cột: {sorted(missing_cols)}")

        relative_path = file_path.relative_to(data_path)
        df["source_file"] = file_path.name
        df["recording_id"] = relative_path.with_suffix("").as_posix()
        df["subject_id"] = file_path.parent.name
        dfs.append(df)

    if not dfs:
        raise ValueError(f"Không tìm thấy file CSV nào tại: {data_path}")
    return pd.concat(dfs, ignore_index=True)


def create_sliding_windows(df, feature_cols, target_col, window_size, step):
    """Create fixed-length windows and label them by the Fall ratio threshold."""
    windows = []
    labels = []
    recording_ids = []
    subject_ids = []

    for recording_id, recording in df.groupby("recording_id", sort=False):
        recording = recording.sort_values("FrameCounter")
        values = recording[feature_cols].to_numpy(dtype=np.float32)
        recording_labels = recording[target_col].to_numpy(dtype=np.int8)
        subject_id = recording["subject_id"].iloc[0]

        if len(recording) < window_size:
            continue

        for start in range(0, len(recording) - window_size + 1, step):
            end = start + window_size
            windows.append(values[start:end].reshape(-1))
            window_labels = recording_labels[start:end]
            fall_ratio = np.mean(window_labels == 1)
            labels.append(int(fall_ratio >= FALL_RATIO_THRESHOLD))
            recording_ids.append(recording_id)
            subject_ids.append(subject_id)

    if not windows:
        raise ValueError("Không tạo được sliding window nào. Kiểm tra window_size.")

    return (
        np.asarray(windows, dtype=np.float32),
        np.asarray(labels, dtype=np.int8),
        np.asarray(recording_ids),
        np.asarray(subject_ids),
    )


In [ ]:
# Đường dẫn dữ liệu huấn luyện: đọc toàn bộ các subject trong Sample_Training
train_path = resolve_dataset_path(
    "Fall_Detection_SIC_Capstone/fall_detection_dataset/Sample_Training",
    "fall_detection_dataset/Sample_Training",
)

# Dùng cùng thư mục output dù chạy từ workspace hay project root.
MODEL_DIR = train_path.parent.parent / "saved_models_3ft"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df_train_all = load_imu_dataset(train_path)

print(f"\nKích thước tập Training: {df_train_all.shape}")
print(f"Số subject: {df_train_all['subject_id'].nunique()}")
print(f"Số recording: {df_train_all['recording_id'].nunique()}")
print("\nSố recording theo subject:")
print(df_train_all.groupby("subject_id")["recording_id"].nunique().describe())

print("\nPhân bố nhãn FallCheck (0: Bình thường, 1: Ngã):")
val_counts = df_train_all["FallCheck"].value_counts().sort_index()
print(val_counts)
print(f"Tỷ lệ mẫu ngã: {val_counts.get(1, 0) / len(df_train_all) * 100:.2f}%")


### 1. Phân chia Train / Validation theo Subject & Chuẩn hóa


In [ ]:
FEATURE_SETS = {
    "A_raw_acc": ["AccX", "AccY", "AccZ"],
}

# Chỉ sử dụng ba trục gia tốc.
selected_feature_set = "A_raw_acc"
feature_cols = FEATURE_SETS[selected_feature_set]
target_col = "FallCheck"

WINDOW_SIZE = 200  # 2 giây x 100 Hz
WINDOW_STEP = 100  # overlap 50%

P_JITTER = 0.30
P_SCALING = 0.30
P_ROTATION = 0.30

ACC_AXIS_NAMES = ["AccX", "AccY", "AccZ"]
ACC_AXIS_INDICES = [feature_cols.index(name) for name in ACC_AXIS_NAMES if name in feature_cols]


def to_window_tensor(X):
    """Convert flattened windows to (samples, time, features)."""
    X = np.asarray(X, dtype=np.float32)
    if X.ndim == 2 and X.shape[1] == WINDOW_SIZE * len(feature_cols):
        return X.reshape(-1, WINDOW_SIZE, len(feature_cols))
    return X


def from_window_tensor(X):
    """Convert window tensors back to flattened model inputs."""
    X = np.asarray(X, dtype=np.float32)
    if X.ndim == 3 and X.shape[1] == WINDOW_SIZE and X.shape[2] == len(feature_cols):
        return X.reshape(X.shape[0], -1)
    return X


def augment_jitter_window(window, sigma=0.05):
    """Add independent Gaussian noise to raw Acc XYZ only."""
    augmented = window.copy()
    if len(ACC_AXIS_INDICES) == 3:
        noise = np.random.normal(0.0, sigma, size=(WINDOW_SIZE, 3)).astype(np.float32)
        augmented[:, ACC_AXIS_INDICES] += noise
    return augmented


def augment_scaling_window(window, sigma=0.10):
    """Scale all Acc XYZ together."""
    augmented = window.copy()
    if len(ACC_AXIS_INDICES) == 3:
        acc_scale = np.float32(np.random.normal(1.0, sigma))
        augmented[:, ACC_AXIS_INDICES] *= acc_scale
    return augmented


def augment_rotation_window(window):
    """Rotate raw Acc XYZ using feature-name-derived indices."""
    augmented = window.copy()
    angle_range = np.pi / 18
    ax, ay, az = np.random.uniform(-angle_range, angle_range, size=3)

    Rx = np.array([[1.0, 0.0, 0.0], [0.0, np.cos(ax), -np.sin(ax)], [0.0, np.sin(ax), np.cos(ax)]], dtype=np.float32)
    Ry = np.array([[np.cos(ay), 0.0, np.sin(ay)], [0.0, 1.0, 0.0], [-np.sin(ay), 0.0, np.cos(ay)]], dtype=np.float32)
    Rz = np.array([[np.cos(az), -np.sin(az), 0.0], [np.sin(az), np.cos(az), 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    rotation = Rz @ Ry @ Rx

    if len(ACC_AXIS_INDICES) == 3:
        augmented[:, ACC_AXIS_INDICES] = augmented[:, ACC_AXIS_INDICES] @ rotation.T
    return augmented


def apply_augmentations(X, y):
    """Augment each raw training window independently; validation/test are never passed here."""
    raw_windows = to_window_tensor(X)
    augmented_windows = raw_windows.copy()
    y_aug = np.asarray(y, dtype=np.int32).copy()

    for sample_index in range(raw_windows.shape[0]):
        window = raw_windows[sample_index].copy()
        if np.random.rand() < P_JITTER:
            window = augment_jitter_window(window)
        if np.random.rand() < P_SCALING:
            window = augment_scaling_window(window)
        if np.random.rand() < P_ROTATION:
            window = augment_rotation_window(window)
        augmented_windows[sample_index] = window

    return from_window_tensor(augmented_windows), y_aug


def scale_windows(X, fitted_scaler):
    """Scale windows using one feature-wise scaler fitted on original raw train windows."""
    tensor = to_window_tensor(X)
    shape = tensor.shape
    scaled = fitted_scaler.transform(tensor.reshape(-1, len(feature_cols)))
    return scaled.reshape(shape).reshape(shape[0], -1).astype(np.float32)


groups = df_train_all["subject_id"].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(df_train_all[feature_cols], df_train_all[target_col], groups=groups))

# Split raw data first. No scaler or augmentation is applied to validation data.
train_df = df_train_all.iloc[train_idx].copy()
val_df = df_train_all.iloc[val_idx].copy()

assert set(train_df["subject_id"].unique()).isdisjoint(set(val_df["subject_id"].unique())), (
    "Leakage: subject overlap detected between train and validation"
)

# Sliding Window -> raw train/validation windows.
X_train_raw, y_train, train_recordings, train_window_subjects = create_sliding_windows(
    train_df, feature_cols, target_col, WINDOW_SIZE, WINDOW_STEP
)
X_val_raw, y_val, val_recordings, val_window_subjects = create_sliding_windows(
    val_df, feature_cols, target_col, WINDOW_SIZE, WINDOW_STEP
)

train_subjects = np.unique(train_window_subjects)
val_subjects = np.unique(val_window_subjects)
assert set(train_subjects).isdisjoint(val_subjects), "Leakage: duplicated subject across windowed train/val"

# Augment only raw training windows. Time shift is intentionally disabled.
X_train_aug_raw, y_train_aug = apply_augmentations(X_train_raw, y_train)

# Fit only on original raw training windows, then transform original/augmented train and validation.
scaler = StandardScaler()
scaler.fit(to_window_tensor(X_train_raw).reshape(-1, len(feature_cols)))
X_train = scale_windows(X_train_raw, scaler)
X_train_augmented = scale_windows(X_train_aug_raw, scaler)
X_val = scale_windows(X_val_raw, scaler)

# Keep the existing variable names expected by later training cells.
X_train_aug = np.concatenate([X_train, X_train_augmented], axis=0)
y_train_aug = np.concatenate([y_train, y_train_aug], axis=0)

print(f"Selected feature set: {selected_feature_set}")
print(f"Features: {feature_cols}")
print(f"Train subjects: {len(train_subjects)} | {train_subjects.tolist()}")
print(f"Val subjects:   {len(val_subjects)} | {val_subjects.tolist()}")
print(f"Original raw train windows: {X_train_raw.shape}")
print(f"Augmented raw train windows: {X_train_aug_raw.shape}")
print(f"Scaled train windows: {X_train_aug.shape} | Tỷ lệ window ngã: {np.mean(y_train_aug == 1) * 100:.2f}%")
print(f"Scaled val windows:   {X_val.shape} | Tỷ lệ window ngã: {np.mean(y_val == 1) * 100:.2f}%")
print(f"Augmentation probabilities: jitter={P_JITTER}, scaling={P_SCALING}, rotation={P_ROTATION}, time_shift=OFF")


### 2. Xử lý mất cân bằng dữ liệu (Class Weight) & Xây dựng mô hình


In [ ]:
# Tính class weight từ dữ liệu train đã augment
classes = np.unique(y_train_aug)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_aug
)
class_weight = dict(zip(classes, weights))
print("Class weights:", class_weight)

In [ ]:
# ============================================================
# 1. SE BLOCK 1D
# ============================================================

def se_block_1d(x, ratio=2):
    """
    Squeeze-and-Excitation attention cho feature map 1D
    có dạng: (batch, time, channels).

    GlobalAveragePooling ở đây dùng để tính channel attention,
    KHÔNG phải global temporal pooling cuối mạng.
    """
    ch = x.shape[-1]

    # Squeeze theo chiều thời gian
    se = layers.GlobalAveragePooling1D()(x)
    se = layers.Reshape((1, ch))(se)

    # Excitation
    se = layers.Conv1D(
        max(ch // ratio, 4),
        kernel_size=1,
        activation="relu",
        use_bias=False
    )(se)

    se = layers.Conv1D(
        ch,
        kernel_size=1,
        activation="sigmoid",
        use_bias=False
    )(se)

    # Channel-wise attention
    return layers.Multiply()([x, se])


# ============================================================
# 2. DEPTHWISE SEPARABLE PLAIN BLOCK 1D
# ============================================================

def ds_block_plain_1d(
    x,
    filters,
    kernel_size=5,
    dilation_rate=1
):
    """
    Depthwise separable Conv1D block.

    dilation_rate giúp tăng temporal receptive field
    mà không cần tăng quá nhiều số parameter.
    """

    # Depthwise temporal convolution
    x = layers.DepthwiseConv1D(
        kernel_size=kernel_size,
        padding="same",
        dilation_rate=dilation_rate,
        use_bias=False
    )(x)

    x = layers.BatchNormalization(
        momentum=0.9
    )(x)

    x = layers.ReLU(6.0)(x)

    # Pointwise convolution: trộn thông tin giữa các channel
    x = layers.Conv1D(
        filters,
        kernel_size=1,
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization(
        momentum=0.9
    )(x)

    x = layers.ReLU(6.0)(x)

    return x


# ============================================================
# 3. DEPTHWISE SEPARABLE RESIDUAL + SE BLOCK 1D
# ============================================================

def ds_se_residual_block_1d(
    x,
    filters,
    stride=1,
    kernel_size=5,
    dilation_rate=1
):
    """
    Residual block sử dụng:
        Depthwise Conv1D
        + Pointwise Conv1D
        + SE Attention
        + Residual connection

    Lưu ý:
    Khi stride=2 thì dilation_rate phải để =1.
    """

    if stride > 1 and dilation_rate > 1:
        raise ValueError(
            "Không dùng đồng thời stride > 1 và dilation_rate > 1."
        )

    in_ch = x.shape[-1]
    residual = x

    # --------------------------------------------------------
    # Main branch
    # --------------------------------------------------------

    y = layers.DepthwiseConv1D(
        kernel_size=kernel_size,
        strides=stride,
        padding="same",
        dilation_rate=dilation_rate,
        use_bias=False
    )(x)

    y = layers.BatchNormalization(
        momentum=0.9
    )(y)

    y = layers.ReLU(6.0)(y)

    # Pointwise convolution
    y = layers.Conv1D(
        filters,
        kernel_size=1,
        padding="same",
        use_bias=False
    )(y)

    y = layers.BatchNormalization(
        momentum=0.9
    )(y)

    # SE attention
    y = se_block_1d(y)


    # --------------------------------------------------------
    # Residual / shortcut branch
    # --------------------------------------------------------

    # Nếu main branch giảm temporal dimension
    if stride == 2:
        residual = layers.AveragePooling1D(
            pool_size=2,
            strides=2,
            padding="same"
        )(residual)

    # Nếu số channel thay đổi
    if in_ch != filters:
        residual = layers.Conv1D(
            filters,
            kernel_size=1,
            padding="same",
            use_bias=False
        )(residual)

        residual = layers.BatchNormalization(
            momentum=0.9
        )(residual)


    # --------------------------------------------------------
    # Residual addition
    # --------------------------------------------------------

    y = layers.Add()([
        y,
        residual
    ])

    y = layers.ReLU(6.0)(y)

    return y


# ============================================================
# 4. XÂY DỰNG MODEL
# ============================================================

inputs = Input(
    shape=(X_train.shape[1],),
    name="input_sensor_features"
)

# Ví dụ:
# 600 features -> (200 timestep, 3 channels)
x = layers.Reshape(
    (WINDOW_SIZE, len(feature_cols)),
    name="reshape_to_time_series"
)(inputs)


# ============================================================
# TEMPORAL FEATURE EXTRACTION
# ============================================================

# ------------------------------------------------------------
# Block 1
# Học các pattern cục bộ ban đầu
#
# kernel = 5
# dilation = 1
# ------------------------------------------------------------

x = ds_block_plain_1d(
    x,
    filters=32,
    kernel_size=5,
    dilation_rate=1
)


# ------------------------------------------------------------
# Block 2
# Bắt đầu mở rộng receptive field theo thời gian
#
# dilation = 2
# ------------------------------------------------------------

x = ds_se_residual_block_1d(
    x,
    filters=64,
    stride=1,
    kernel_size=5,
    dilation_rate=2
)


# ------------------------------------------------------------
# Block 3
# Downsampling temporal dimension
#
# 200 timestep -> 100 timestep
#
# Không dùng dilation > 1 khi stride = 2
# ------------------------------------------------------------

x = ds_se_residual_block_1d(
    x,
    filters=64,
    stride=2,
    kernel_size=5,
    dilation_rate=1
)


# ------------------------------------------------------------
# Block 4
# Temporal context rộng hơn
# ------------------------------------------------------------

x = ds_se_residual_block_1d(
    x,
    filters=64,
    stride=1,
    kernel_size=5,
    dilation_rate=2
)


# ------------------------------------------------------------
# Block 5
# Temporal receptive field rộng hơn nữa
#
# dilation = 4
# ------------------------------------------------------------

x = ds_se_residual_block_1d(
    x,
    filters=64,
    stride=1,
    kernel_size=5,
    dilation_rate=4
)


# ============================================================
# 5. GLOBAL TEMPORAL POOLING
# ============================================================

# Average pooling:
# nắm thông tin tổng thể của cả window
x_avg = layers.GlobalAveragePooling1D(
    name="global_avg_pool"
)(x)

# Max pooling:
# giữ lại transient / activation mạnh nhất,
# rất hữu ích với sự kiện Fall ngắn
x_max = layers.GlobalMaxPooling1D(
    name="global_max_pool"
)(x)

# Kết hợp cả global context và transient peak
x = layers.Concatenate(
    name="avg_max_pool_concat"
)([
    x_avg,
    x_max
])


# ============================================================
# 6. CLASSIFIER
# ============================================================

x = layers.Dense(
    64,
    activation="relu",
    name="dense_64"
)(x)

x = layers.BatchNormalization(
    name="bn_dense_64"
)(x)

x = layers.Dropout(
    0.35,
    name="dropout_035"
)(x)


x = layers.Dense(
    32,
    activation="relu",
    name="dense_32"
)(x)

x = layers.BatchNormalization(
    name="bn_dense_32"
)(x)

x = layers.Dropout(
    0.20,
    name="dropout_020"
)(x)


# ============================================================
# 7. OUTPUT
# ============================================================

outputs = layers.Dense(
    1,
    activation="sigmoid",
    name="fall_prediction"
)(x)


model = Model(
    inputs=inputs,
    outputs=outputs,
    name="Fall_Detection_1D_CNN_SE"
)


# ============================================================
# 8. COMPILE
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="binary_crossentropy",

    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy", threshold=DECISION_THRESHOLD
        ),

        tf.keras.metrics.Precision(
            name="precision", thresholds=DECISION_THRESHOLD
        ),

        tf.keras.metrics.Recall(
            name="recall", thresholds=DECISION_THRESHOLD
        ),

        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)


# ============================================================
# 9. MODEL SUMMARY
# ============================================================

model.summary()

### 3. Huấn luyện mô hình với ngưỡng phân loại cố định 0.5

Chọn checkpoint theo `val_recall` tại ngưỡng cố định.


In [ ]:
best_model_path = str(MODEL_DIR / "best_model.keras")

callbacks = [
    ModelCheckpoint(
        filepath=best_model_path,
        monitor="val_recall",
        mode="max",
        save_best_only=True,
        save_weights_only=False,
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_recall",
        mode="max",
        patience=12,
        min_delta=1e-3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]

history = model.fit(
    X_train_aug,
    y_train_aug,
    validation_data=(X_val, y_val),
    batch_size=32,
    epochs=100,
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)

loss_history = np.asarray(history.history["loss"])
val_loss_history = np.asarray(history.history["val_loss"])
val_recall_history = np.asarray(history.history["val_recall"])

best_recall_epoch = int(np.argmax(val_recall_history) + 1)
best_loss_epoch = int(np.argmin(val_loss_history) + 1)
stop_epoch = len(val_loss_history)

print(f"Best val_recall: epoch {best_recall_epoch} ({val_recall_history[best_recall_epoch - 1]:.6f})")
print(f"Best val_loss: epoch {best_loss_epoch} ({val_loss_history[best_loss_epoch - 1]:.6f})")
print(f"Val recall at best val_loss epoch: {val_recall_history[best_loss_epoch - 1]:.6f}")
print(f"Val recall at best val_recall epoch: {val_recall_history[best_recall_epoch - 1]:.6f}")
print(f"Stopped after: epoch {stop_epoch}")
print(f"Train loss at best val_recall epoch: {loss_history[best_recall_epoch - 1]:.6f}")
print(f"Best model saved during training at: {best_model_path}")
print("Lưu ý: train loss có class_weight, còn val_loss và val_recall mặc định không có class_weight.")

# Đánh giá đúng checkpoint đã lưu, cùng checkpoint dùng để fine-tune.
model = tf.keras.models.load_model(best_model_path)
# Confusion matrix trên Validation tại ngưỡng cố định.
val_probs = model.predict(X_val, verbose=0).ravel()
val_preds = (val_probs > DECISION_THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()

print(f"\nConfusion Matrix trên Validation (ngưỡng cố định = {DECISION_THRESHOLD})")
print(f"TP (Fall -> Fall):     {tp}")
print(f"FN (Fall -> Normal):   {fn}")
print(f"FP (Normal -> Fall):   {fp}")
print(f"TN (Normal -> Normal): {tn}")
print(f"Recall tại ngưỡng {DECISION_THRESHOLD}: {recall_score(y_val, val_preds, zero_division=0):.6f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["loss"], label="Train Loss", color="blue")
axes[0].plot(history.history["val_loss"], label="Val Loss", color="red", linestyle="--")
axes[0].axvline(best_loss_epoch - 1, color="black", linestyle=":", label=f"Best val_loss = {best_loss_epoch}")
axes[0].axvline(best_recall_epoch - 1, color="purple", linestyle="-.", label=f"Best val_recall = {best_recall_epoch}")
axes[0].set_title("Quá trình Biến thiên Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history["recall"], label="Train Recall", color="green")
axes[1].plot(history.history["val_recall"], label="Val Recall", color="orange", linestyle="--")
axes[1].axvline(best_recall_epoch - 1, color="purple", linestyle="-.", label=f"Best val_recall = {best_recall_epoch}")
axes[1].axvline(best_loss_epoch - 1, color="black", linestyle=":", label=f"Best val_loss = {best_loss_epoch}")
axes[1].set_title("Quá trình Biến thiên Recall")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Recall")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 4. Fine-tuning và đánh giá Validation tại ngưỡng cố định 0.5

Tiếp tục huấn luyện từ checkpoint tốt nhất với learning rate nhỏ hơn.
Model fine-tuned được chọn theo `val_recall`, rồi dùng thống nhất cho đánh giá test và export.


In [ ]:
# ============================================================
# FINE-TUNING MODEL
# ============================================================

# ------------------------------------------------------------
# 1. Load lại checkpoint tốt nhất
# ------------------------------------------------------------

print("Load best model:", best_model_path)

model_ft = tf.keras.models.load_model(
    best_model_path
)

# ------------------------------------------------------------
# 2. Compile lại với learning rate nhỏ hơn
# ------------------------------------------------------------
# Training ban đầu: lr = 1e-3
# Fine-tuning:      lr = 1e-4

model_ft.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy", threshold=DECISION_THRESHOLD
        ),

        tf.keras.metrics.Precision(
            name="precision", thresholds=DECISION_THRESHOLD
        ),

        tf.keras.metrics.Recall(
            name="recall", thresholds=DECISION_THRESHOLD
        ),

        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)

# ------------------------------------------------------------
# 3. Callback cho fine-tuning
# ------------------------------------------------------------

fine_tuned_model_path = str(MODEL_DIR / "fall_detection_finetuned.keras")

callbacks_ft = [

    # Vẫn ưu tiên khả năng bắt Fall
    ModelCheckpoint(
        filepath=fine_tuned_model_path,
        monitor="val_recall",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    # Fine-tuning không cần train quá lâu
    EarlyStopping(
        monitor="val_recall",
        mode="max",
        patience=6,
        min_delta=1e-3,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

# ------------------------------------------------------------
# 4. Fine-tune
# ------------------------------------------------------------

history_ft = model_ft.fit(
    X_train_aug,
    y_train_aug,

    validation_data=(
        X_val,
        y_val
    ),

    batch_size=32,

    # Chỉ fine-tune thêm một số epoch nhỏ
    epochs=20,

    callbacks=callbacks_ft,

    class_weight=class_weight,

    verbose=1
)

# ------------------------------------------------------------
# 5. Load lại checkpoint tốt nhất của fine-tuning
# ------------------------------------------------------------

model_ft = tf.keras.models.load_model(
    fine_tuned_model_path
)

print("\nFine-tuning hoàn tất.")
print("Best fine-tuned model:", fine_tuned_model_path)

# ------------------------------------------------------------
# 6. Đánh giá Validation ở ngưỡng cố định 0.5
# ------------------------------------------------------------

results = model_ft.evaluate(
    X_val,
    y_val,
    verbose=0,
    return_dict=True
)

print(f"\nValidation metrics (ngưỡng cố định = {DECISION_THRESHOLD}):")
for name, value in results.items():
    print(f"{name}: {value:.4f}")

val_ft_probs = model_ft.predict(X_val, verbose=0).ravel()
val_ft_preds = (val_ft_probs > DECISION_THRESHOLD).astype(int)
print(classification_report(
    y_val, val_ft_preds, labels=[0, 1],
    target_names=["Bình thường (0)", "Ngã (1)"], digits=4, zero_division=0
))
ConfusionMatrixDisplay.from_predictions(
    y_val, val_ft_preds, labels=[0, 1],
    display_labels=["Bình thường", "Ngã"], cmap="Blues", values_format="d"
)
plt.title(f"Validation - Fine-tuned model (ngưỡng cố định = {DECISION_THRESHOLD})")
plt.tight_layout()
plt.show()


### 5. Đánh giá model fine-tuned trên toàn bộ Sample_Test

Dùng scaler đã fit trên train và ngưỡng cố định 0.5. Báo cáo theo subject và toàn bộ test;
không dùng test để chọn checkpoint hoặc ngưỡng phân loại.


In [ ]:
# ============================================================
# ĐÁNH GIÁ TOÀN BỘ SAMPLE_TEST
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ============================================================
# 1. ĐƯỜNG DẪN THƯ MỤC SAMPLE_TEST
# ============================================================

test_root = resolve_dataset_path(
    "Fall_Detection_SIC_Capstone/fall_detection_dataset/Sample_Test",
    "fall_detection_dataset/Sample_Test",
)

print("Test root:", test_root)


# ============================================================
# 2. TÌM TẤT CẢ SUBJECT TEST: SA21, SA23, ...
# ============================================================

test_subject_dirs = sorted(
    [
        p for p in Path(test_root).iterdir()
        if p.is_dir() and p.name.startswith("SA")
    ],
    key=lambda p: p.name
)

if len(test_subject_dirs) == 0:
    raise ValueError(
        f"Không tìm thấy folder SAxx nào trong {test_root}"
    )

print(
    "Các subject test:",
    [p.name for p in test_subject_dirs]
)


# ============================================================
# 3. BIẾN LƯU KẾT QUẢ TOÀN BỘ TEST
# ============================================================

all_y_true = []
all_y_pred = []
all_y_prob = []

all_recordings = []
all_subjects = []

subject_results = []


# ============================================================
# 4. TEST TỪNG SUBJECT
# ============================================================

for subject_dir in test_subject_dirs:

    subject_name = subject_dir.name

    print("\n" + "=" * 65)
    print(f" ĐANG ĐÁNH GIÁ SUBJECT: {subject_name}")
    print("=" * 65)

    # --------------------------------------------------------
    # Load raw IMU data của subject
    # --------------------------------------------------------

    df_test = load_imu_dataset(
        subject_dir
    )

    print(
        f"Kích thước raw data: {df_test.shape}"
    )


    # Tạo window raw rồi chuẩn hóa như train/validation, theo từng recording.
    X_test_raw, y_test, test_recordings, test_window_subjects = create_sliding_windows(
        df_test, feature_cols, target_col, WINDOW_SIZE, WINDOW_STEP
    )
    X_test = scale_windows(X_test_raw, scaler)

    print(
        f"Số window: {len(X_test)}"
    )

    print(
        f"Normal: {np.sum(y_test == 0)} | "
        f"Fall: {np.sum(y_test == 1)}"
    )


    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    test_probs = model_ft.predict(
        X_test,
        verbose=0
    ).ravel()

    y_pred = (
        test_probs > DECISION_THRESHOLD
    ).astype(int)


    # ========================================================
    # METRICS SUBJECT
    # ========================================================

    acc = accuracy_score(
        y_test,
        y_pred
    )

    precision_fall = precision_score(
        y_test,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    recall_fall = recall_score(
        y_test,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    f1_fall = f1_score(
        y_test,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    f1_macro = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )


    # ROC-AUC chỉ tính được nếu subject có cả 2 lớp
    if len(np.unique(y_test)) == 2:

        auc = roc_auc_score(
            y_test,
            test_probs
        )

    else:

        auc = np.nan


    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    ).ravel()


    print(
        f"\nAccuracy       : {acc:.4f}"
    )

    print(
        f"Precision Fall : {precision_fall:.4f}"
    )

    print(
        f"Recall Fall    : {recall_fall:.4f}"
    )

    print(
        f"F1 Fall        : {f1_fall:.4f}"
    )

    print(
        f"F1 Macro       : {f1_macro:.4f}"
    )

    if not np.isnan(auc):
        print(
            f"ROC-AUC        : {auc:.4f}"
        )

    print(
        f"TN={tn} | FP={fp} | FN={fn} | TP={tp}"
    )


    # --------------------------------------------------------
    # Lưu metric subject
    # --------------------------------------------------------

    subject_results.append({

        "Subject": subject_name,

        "Windows": len(y_test),

        "Normal": int(
            np.sum(y_test == 0)
        ),

        "Fall": int(
            np.sum(y_test == 1)
        ),

        "Accuracy": acc,

        "Precision_Fall": precision_fall,

        "Recall_Fall": recall_fall,

        "F1_Fall": f1_fall,

        "F1_Macro": f1_macro,

        "ROC_AUC": auc,

        "TN": int(tn),

        "FP": int(fp),

        "FN": int(fn),

        "TP": int(tp),
    })


    # --------------------------------------------------------
    # Gộp prediction SAU KHI đã window riêng
    # --------------------------------------------------------

    all_y_true.append(
        y_test
    )

    all_y_pred.append(
        y_pred
    )

    all_y_prob.append(
        test_probs
    )

    all_recordings.append(
        test_recordings
    )

    all_subjects.append(
        np.full(
            len(y_test),
            subject_name
        )
    )


# ============================================================
# 5. GỘP TOÀN BỘ TEST WINDOWS
# ============================================================

all_y_true = np.concatenate(
    all_y_true
)

all_y_pred = np.concatenate(
    all_y_pred
)

all_y_prob = np.concatenate(
    all_y_prob
)

all_recordings = np.concatenate(
    all_recordings
)

all_subjects = np.concatenate(
    all_subjects
)


# ============================================================
# 6. BẢNG KẾT QUẢ TỪNG SUBJECT
# ============================================================

results_df = pd.DataFrame(
    subject_results
)

print("\n\n" + "=" * 90)
print(" KẾT QUẢ TỪNG SUBJECT")
print("=" * 90)

display(
    results_df.round(4)
)


# ============================================================
# 7. METRICS TỔNG TOÀN BỘ SAMPLE_TEST
# ============================================================

overall_acc = accuracy_score(
    all_y_true,
    all_y_pred
)

overall_precision = precision_score(
    all_y_true,
    all_y_pred,
    pos_label=1,
    zero_division=0
)

overall_recall = recall_score(
    all_y_true,
    all_y_pred,
    pos_label=1,
    zero_division=0
)

overall_f1 = f1_score(
    all_y_true,
    all_y_pred,
    pos_label=1,
    zero_division=0
)

overall_f1_macro = f1_score(
    all_y_true,
    all_y_pred,
    average="macro",
    zero_division=0
)

overall_auc = (
    roc_auc_score(all_y_true, all_y_prob)
    if len(np.unique(all_y_true)) == 2 else np.nan
)

tn, fp, fn, tp = confusion_matrix(
    all_y_true,
    all_y_pred,
    labels=[0, 1]
).ravel()


# F2 vì bài toán ưu tiên Recall
beta = 2

overall_f2 = (
    (1 + beta**2)
    * overall_precision
    * overall_recall
    /
    (
        beta**2 * overall_precision
        + overall_recall
        + 1e-8
    )
)


# ============================================================
# 8. IN KẾT QUẢ TỔNG
# ============================================================

print("\n" + "=" * 65)
print(" KẾT QUẢ TỔNG TOÀN BỘ SAMPLE_TEST")
print("=" * 65)

print(
    f"Số subjects     : {len(test_subject_dirs)}"
)

print(
    f"Tổng windows    : {len(all_y_true)}"
)

print(
    f"Normal windows  : {np.sum(all_y_true == 0)}"
)

print(
    f"Fall windows    : {np.sum(all_y_true == 1)}"
)

print()

print(
    f"Accuracy        : {overall_acc:.4f}"
)

print(
    f"Precision Fall  : {overall_precision:.4f}"
)

print(
    f"Recall Fall     : {overall_recall:.4f}"
)

print(
    f"F1 Fall         : {overall_f1:.4f}"
)

print(
    f"F2 Fall         : {overall_f2:.4f}"
)

print(
    f"F1 Macro        : {overall_f1_macro:.4f}"
)

print(
    f"ROC-AUC         : {overall_auc:.4f}"
)

print()

print(
    f"TN: {tn}"
)

print(
    f"FP: {fp}"
)

print(
    f"FN: {fn}"
)

print(
    f"TP: {tp}"
)


# ============================================================
# 9. CLASSIFICATION REPORT TỔNG
# ============================================================

print(
    "\nClassification Report - ALL TEST SUBJECTS:"
)

print(
    classification_report(
        all_y_true,
        all_y_pred,
        labels=[0, 1],
        target_names=[
            "Bình thường (0)",
            "Ngã (1)"
        ],
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 10. CONFUSION MATRIX TỔNG
# ============================================================

fig, ax = plt.subplots(
    figsize=(6, 5)
)

cm = confusion_matrix(
    all_y_true,
    all_y_pred,
    labels=[0, 1]
)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Bình thường",
        "Ngã"
    ]
).plot(
    cmap="Blues",
    values_format="d",
    ax=ax
)

ax.set_title(
    "Confusion Matrix - ALL Sample_Test\n"
    f"Threshold = {DECISION_THRESHOLD:.2f} (cố định)"
)

plt.tight_layout()
plt.show()


# ============================================================
# 11. MACRO AVERAGE THEO SUBJECT
# ============================================================

print("\n" + "=" * 65)
print(" TRUNG BÌNH THEO SUBJECT")
print("=" * 65)

print(
    f"Mean Accuracy       : "
    f"{results_df['Accuracy'].mean():.4f}"
)

print(
    f"Mean Precision Fall : "
    f"{results_df['Precision_Fall'].mean():.4f}"
)

print(
    f"Mean Recall Fall    : "
    f"{results_df['Recall_Fall'].mean():.4f}"
)

print(
    f"Mean F1 Fall        : "
    f"{results_df['F1_Fall'].mean():.4f}"
)

print(
    f"Mean ROC-AUC        : "
    f"{results_df['ROC_AUC'].mean():.4f}"
)

### 6. Xuất model, scaler và cấu hình triển khai

Xuất checkpoint fine-tuned đã đánh giá ở trên. Metadata và C/C++ dùng cùng ngưỡng cố định 0.5,
đầu vào 600 giá trị và scaler theo 3 trục gia tốc.


In [ ]:
# ============================================================
# CONVERT FINE-TUNED KERAS MODEL -> TFLITE
# ============================================================

import os
import tensorflow as tf

KERAS_PATH = fine_tuned_model_path
TFLITE_PATH = str(MODEL_DIR / "fall_detection_finetuned.tflite")

# 1. Load model fine-tuned
model_export = tf.keras.models.load_model(
    KERAS_PATH
)

print("Loaded:", KERAS_PATH)

# 2. Convert sang TFLite Float32
converter = tf.lite.TFLiteConverter.from_keras_model(
    model_export
)

tflite_model = converter.convert()

# 3. Lưu vào MODEL_DIR
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

print("\nTFLite export thành công!")
print("Path:", os.path.abspath(TFLITE_PATH))
print(
    "Size:",
    round(os.path.getsize(TFLITE_PATH) / 1024, 2),
    "KB"
)

# 4. Kiểm tra danh sách file
print(f"\nFiles trong {MODEL_DIR}:")
for file in sorted(os.listdir(MODEL_DIR)):
    print(" -", file)

In [ ]:
# LƯU FILE PKL: SCALER VÀ METADATA (ACC XYZ)
# Chạy sau chuẩn hóa và export TFLite, trước cell export C/C++.
from pathlib import Path
import joblib

pkl_dir = Path(MODEL_DIR)

if "scaler" not in globals() or not hasattr(scaler, "mean_"):
    raise RuntimeError("Hãy chạy cell chuẩn hóa để có scaler đã fit trước khi lưu.")
if "feature_cols" not in globals() or list(feature_cols) != ["AccX", "AccY", "AccZ"]:
    raise ValueError("Cell này lưu cấu hình 3 đặc trưng AccX, AccY, AccZ.")
if len(scaler.mean_) != len(feature_cols):
    raise ValueError("Scaler không khớp 3 đặc trưng AccX, AccY, AccZ.")

metadata = {
    "feature_cols": list(feature_cols),
    "window_size": int(WINDOW_SIZE),
    "window_step": int(WINDOW_STEP),
    "decision_threshold": float(DECISION_THRESHOLD),
    "decision_operator": ">",
    "fall_ratio_threshold": float(FALL_RATIO_THRESHOLD),
    "scaler_mode": "per_feature",
}

pkl_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, pkl_dir / "scaler.pkl")
joblib.dump(metadata, pkl_dir / "metadata.pkl")

print("Đã lưu:", (pkl_dir / "scaler.pkl").resolve())
print("Đã lưu:", (pkl_dir / "metadata.pkl").resolve())


In [ ]:
# ============================================================
# EXPORT TFLITE + SCALER -> C/C++ FILES
#
# Output:
#   saved_models_3ft/model_data.cc
#   saved_models_3ft/model_data.h
#   saved_models_3ft/scaler_data.h
# ============================================================

import os
import joblib
import numpy as np


# ============================================================
# 1. PATH CONFIG
# ============================================================

TFLITE_PATH = os.path.join(
    MODEL_DIR,
    "fall_detection_finetuned.tflite"
)

SCALER_PATH = os.path.join(
    MODEL_DIR,
    "scaler.pkl"
)

METADATA_PATH = os.path.join(
    MODEL_DIR,
    "metadata.pkl"
)

MODEL_CC_PATH = os.path.join(
    MODEL_DIR,
    "model_data.cc"
)

MODEL_H_PATH = os.path.join(
    MODEL_DIR,
    "model_data.h"
)

SCALER_H_PATH = os.path.join(
    MODEL_DIR,
    "scaler_data.h"
)


# ============================================================
# 2. CHECK INPUT FILES
# ============================================================

required_files = [
    TFLITE_PATH,
    SCALER_PATH,
    METADATA_PATH
]

for path in required_files:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Không tìm thấy file: {path}"
        )

print("Found TFLite :", TFLITE_PATH)
print("Found Scaler :", SCALER_PATH)


# ============================================================
# 3. LOAD SCALER
# ============================================================

scaler_export = joblib.load(
    SCALER_PATH
)

scaler_mean = np.asarray(
    scaler_export.mean_,
    dtype=np.float32
).reshape(-1)

scaler_scale = np.asarray(
    scaler_export.scale_,
    dtype=np.float32
).reshape(-1)

if len(scaler_mean) != len(scaler_scale):
    raise ValueError(
        "scaler.mean_ và scaler.scale_ không cùng kích thước."
    )

SCALER_SIZE = len(scaler_mean)

print(
    "Scaler size:",
    SCALER_SIZE
)


# ============================================================
# 4. LOAD METADATA VÀ KIỂM TRA CẤU HÌNH EXPORT
# ============================================================

metadata_export = joblib.load(METADATA_PATH)
if not isinstance(metadata_export, dict):
    raise ValueError("metadata.pkl phải là dict. Hãy chạy lại cell lưu metadata.")

EXPORT_THRESHOLD = float(DECISION_THRESHOLD)
if (
    metadata_export.get("decision_threshold") != EXPORT_THRESHOLD
    or metadata_export.get("decision_operator") != ">"
):
    raise ValueError("Metadata không khớp ngưỡng cố định. Hãy chạy lại cell lưu metadata.")

EXPORT_WINDOW_SIZE = int(metadata_export["window_size"])
EXPORT_WINDOW_STEP = int(metadata_export["window_step"])
EXPORT_FEATURES = list(metadata_export["feature_cols"])
NUM_FEATURES = len(EXPORT_FEATURES)
EXPECTED_INPUT_SIZE = EXPORT_WINDOW_SIZE * NUM_FEATURES

# Scaler có 3 giá trị (mỗi trục), input có 200 × 3 = 600 giá trị.
if metadata_export.get("scaler_mode") != "per_feature" or SCALER_SIZE != NUM_FEATURES:
    raise ValueError("Scaler phải có một mean/scale cho mỗi đặc trưng.")
if EXPORT_FEATURES != ["AccX", "AccY", "AccZ"]:
    raise ValueError("Thứ tự đặc trưng phải là AccX, AccY, AccZ.")
if not np.all(np.isfinite(scaler_mean)) or not np.all(np.isfinite(scaler_scale)) or np.any(scaler_scale <= 0):
    raise ValueError("Scaler chứa giá trị không hợp lệ.")

print("Window size:", EXPORT_WINDOW_SIZE)
print("Window step:", EXPORT_WINDOW_STEP)
print("Features:", EXPORT_FEATURES)
print("Input size:", EXPECTED_INPUT_SIZE)
print("Scaler size:", SCALER_SIZE)
print("Ngưỡng phân loại cố định:", EXPORT_THRESHOLD)


# ============================================================
# 5. READ TFLITE BINARY
# ============================================================

with open(
    TFLITE_PATH,
    "rb"
) as f:

    model_bytes = f.read()

MODEL_SIZE = len(
    model_bytes
)

print(
    "\nTFLite model size:",
    MODEL_SIZE,
    "bytes"
)

print(
    "TFLite model size:",
    round(
        MODEL_SIZE / 1024,
        2
    ),
    "KB"
)


# ============================================================
# 6. GENERATE model_data.h
# ============================================================

model_h = """\
#ifndef MODEL_DATA_H_
#define MODEL_DATA_H_

#include <cstddef>
#include <cstdint>

// TensorFlow Lite model embedded as a byte array.
extern const unsigned char g_fall_model[];
extern const unsigned int g_fall_model_len;

#endif  // MODEL_DATA_H_
"""


with open(
    MODEL_H_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        model_h
    )


# ============================================================
# 7. GENERATE model_data.cc
# ============================================================

BYTES_PER_LINE = 12

lines = []

for i in range(
    0,
    MODEL_SIZE,
    BYTES_PER_LINE
):

    chunk = model_bytes[
        i:i + BYTES_PER_LINE
    ]

    byte_string = ", ".join(
        f"0x{b:02x}"
        for b in chunk
    )

    lines.append(
        "    " + byte_string
    )


model_array_text = ",\n".join(
    lines
)


model_cc = f"""\
#include "model_data.h"

// Alignment is useful for TensorFlow Lite Micro.
alignas(16) const unsigned char g_fall_model[] = {{
{model_array_text}
}};

const unsigned int g_fall_model_len =
    sizeof(g_fall_model);
"""


with open(
    MODEL_CC_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        model_cc
    )


# ============================================================
# 8. HELPER: FORMAT FLOAT ARRAY
# ============================================================

def format_float_array(
    values,
    values_per_line=6
):

    values = np.asarray(
        values
    ).reshape(-1)

    output_lines = []

    for i in range(
        0,
        len(values),
        values_per_line
    ):

        chunk = values[
            i:i + values_per_line
        ]

        line = ", ".join(
            f"{float(v):.9e}f"
            for v in chunk
        )

        output_lines.append(
            "    " + line
        )

    return ",\n".join(
        output_lines
    )


mean_text = format_float_array(
    scaler_mean
)

scale_text = format_float_array(
    scaler_scale
)


# ============================================================
# 9. FEATURE ORDER COMMENT
# ============================================================

if EXPORT_FEATURES:

    feature_comment = "\n".join(
        f"//   {i}: {name}"
        for i, name in enumerate(
            EXPORT_FEATURES
        )
    )

else:

    feature_comment = (
        "// Feature order unavailable."
    )


# ============================================================
# 10. GENERATE scaler_data.h
# ============================================================

scaler_h = f"""\
#ifndef SCALER_DATA_H_
#define SCALER_DATA_H_

#include <cstddef>

// ============================================================
// FALL DETECTION CONFIGURATION
// ============================================================

constexpr int kWindowSize = {EXPORT_WINDOW_SIZE};
constexpr int kWindowStep = {EXPORT_WINDOW_STEP};
constexpr int kNumFeatures = {NUM_FEATURES};
constexpr int kInputSize = {EXPECTED_INPUT_SIZE};

// Probability > kFallThreshold -> Fall (fixed threshold)
constexpr float kFallThreshold = {EXPORT_THRESHOLD:.9g}f;


// ============================================================
// FEATURE ORDER
// ============================================================

{feature_comment}


// ============================================================
// STANDARD SCALER
//
// Python preprocessing:
//     feature_index = i % kNumFeatures;
//     x_scaled[i] =
//         (x[i] - scaler.mean_[feature_index])
//         / scaler.scale_[feature_index];
//
// Firmware phải thực hiện chính xác cùng phép biến đổi.
// ============================================================

constexpr float kScalerMean[{SCALER_SIZE}] = {{
{mean_text}
}};


constexpr float kScalerScale[{SCALER_SIZE}] = {{
{scale_text}
}};


// ============================================================
// SCALING HELPER
// ============================================================

inline float StandardizeInput(
    float value,
    int index
) {{
    const int feature_index = index % kNumFeatures;
    return (
        value - kScalerMean[feature_index]
    ) / kScalerScale[feature_index];
}}

#endif  // SCALER_DATA_H_
"""


with open(
    SCALER_H_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        scaler_h
    )


# ============================================================
# 11. RESULT
# ============================================================

print("\n" + "=" * 60)
print(" EXPORT C/C++ SUCCESS")
print("=" * 60)

for path in [
    MODEL_CC_PATH,
    MODEL_H_PATH,
    SCALER_H_PATH
]:

    print(
        f"{path:<35} "
        f"{os.path.getsize(path) / 1024:.2f} KB"
    )


print("\nGenerated files:")

print(
    "1.",
    MODEL_CC_PATH
)

print(
    "2.",
    MODEL_H_PATH
)

print(
    "3.",
    SCALER_H_PATH
)

print("\nThreshold exported:")
print(
    f"kFallThreshold = "
    f"{EXPORT_THRESHOLD:.6f}"
)

print("\nDone.")